
# Sampling and Simulation Analysis

This notebook implements the following:
- SRS (without replacement)
- Stratified sampling (proportional and Neyman allocation)
- Monte Carlo simulation (B=1000)
- Sensitivity analysis (price cap and outlier exclusion)

Ratio estimation is intentionally omitted due to weak auxiliary relationship from EDA.



## 1) Setup


In [1]:

import numpy as np
import pandas as pd

from sampling_core import HAVE_SCIPY

RNG_SEED = 407
rng = np.random.default_rng(RNG_SEED)

DATA_PATH = "analysis_dataset.csv"
STRATA_COL = "neighbourhood"
Y_COL = "price"
OUTLIER_FLAG_COL = "price_outlier_iqr"

df = pd.read_csv(DATA_PATH)
print(f"Loaded rows: {len(df)}")
print(df.columns.tolist())


Loaded rows: 4702
['id', 'host_id', 'price', 'neighbourhood', 'room_type', 'number_of_reviews', 'property_type', 'accommodates', 'bedrooms', 'bathrooms', 'price_raw', 'price_outlier_iqr']



## 2) Population definitions

We use the finite population from EDA-aligned `analysis_dataset.csv`.

Checks:
- \(N = 4702\)
- strata from `neighbourhood`
- \(\sum_h N_h = N\)


In [2]:

assert Y_COL in df.columns
assert STRATA_COL in df.columns

population = df.copy()
N = len(population)

stratum_stats = (
    population.groupby(STRATA_COL)[Y_COL]
    .agg(N_h="size", mean_h="mean", S_h="std")
    .reset_index()
    .sort_values("N_h", ascending=False)
)
stratum_stats["W_h"] = stratum_stats["N_h"] / N

print(f"Finite population N = {N}")
print(f"Number of strata = {len(stratum_stats)}")
print(f"Sum N_h = {stratum_stats['N_h'].sum()}")
print(stratum_stats.head(8))


Finite population N = 4702
Number of strata = 23
Sum N_h = 4702
               neighbourhood   N_h      mean_h          S_h       W_h
1                   Downtown  1180  300.435593  1241.834272  0.250957
10                 Kitsilano   321  242.940810   243.389014  0.068269
12            Mount Pleasant   306  165.516340   126.441750  0.065079
2          Downtown Eastside   297  231.636364   320.129971  0.063165
6           Hastings-Sunrise   268  156.358209   127.067230  0.056997
21                  West End   265  217.332075   193.408255  0.056359
7   Kensington-Cedar Cottage   254  193.031496   278.280915  0.054020
15                Riley Park   204  199.901961   195.557382  0.043386



## 3) Estimators and allocation helpers

### SRS formulas
- Mean estimator: \(\hat{ar{Y}} = ar{y}\)
- Variance estimator with FPC: \(\widehat{Var}(ar{y}) = (1-f)\,s^2/n\), where \(f=n/N\)
- CI: \(ar{y} \pm t_{\alpha/2,\,df}\cdot SE\) for small-sample settings

### Stratified formulas
- Combined mean: \(\hat{ar{Y}}_{st} = \sum_h W_har{y}_h\), with \(W_h=N_h/N\)
- Variance estimator: \(\widehat{Var}(\hat{ar{Y}}_{st}) = \sum_h W_h^2 (1-f_h) s_h^2 / n_h\), where \(f_h=n_h/N_h\)


In [3]:

from sampling_core import (
    draw_stratified_sample,
    neyman_allocation,
    proportional_allocation,
    run_all_sensitivity_scenarios,
    run_monte_carlo,
    print_monte_carlo_summary,
    srs_estimate,
    stratified_estimate,
)



## 4) Allocation and one real sample run (n=300)

**Design effect (estimated variances):** \(\mathrm{DEFF} = \widehat{V}_{\text{complex}}(\hat{\bar{Y}}) / \widehat{V}_{\text{SRS}}(\hat{\bar{Y}})\). Values **above 1** mean the complex design has **larger** estimated variance than SRS in that draw; values **below 1** mean **smaller** estimated variance.


In [4]:

n_total = 300

alloc_prop = proportional_allocation(stratum_stats, n_total=n_total, min_per_stratum=2)
alloc_neyman = neyman_allocation(stratum_stats, n_total=n_total, min_per_stratum=2)

alloc_df = stratum_stats[[STRATA_COL, "N_h", "S_h"]].copy()
alloc_df["n_prop"] = alloc_df[STRATA_COL].map(alloc_prop)
alloc_df["n_neyman"] = alloc_df[STRATA_COL].map(alloc_neyman)
alloc_df["target_prop_share"] = alloc_df["N_h"] / alloc_df["N_h"].sum()
alloc_df["achieved_prop_share"] = alloc_df["n_prop"] / n_total
alloc_df["target_neyman_share"] = (alloc_df["N_h"] * alloc_df["S_h"]) / (alloc_df["N_h"] * alloc_df["S_h"]).sum()
alloc_df["achieved_neyman_share"] = alloc_df["n_neyman"] / n_total

print("Allocation checks")
print(f"sum n_prop = {alloc_prop.sum()}, sum n_neyman = {alloc_neyman.sum()}")
print(f"min n_prop = {alloc_prop.min()}, min n_neyman = {alloc_neyman.min()}")
print(alloc_df.sort_values("N_h", ascending=False).head(10))


Allocation checks
sum n_prop = 300, sum n_neyman = 300
min n_prop = 3, min n_neyman = 2
               neighbourhood   N_h          S_h  n_prop  n_neyman  \
1                   Downtown  1180  1241.834272      66       175   
10                 Kitsilano   321   243.389014      19        11   
12            Mount Pleasant   306   126.441750      19         7   
2          Downtown Eastside   297   320.129971      18        13   
6           Hastings-Sunrise   268   127.067230      16         6   
21                  West End   265   193.408255      16         8   
7   Kensington-Cedar Cottage   254   278.280915      16        10   
15                Riley Park   204   195.557382      13         7   
14       Renfrew-Collingwood   204    88.334831      13         4   
3          Dunbar Southlands   171   146.842815      11         5   

    target_prop_share  achieved_prop_share  target_neyman_share  \
1            0.250957             0.220000             0.680914   
10           0.068

In [5]:

# Draw one sample per design
sample_srs_idx = rng.choice(population.index.to_numpy(), size=n_total, replace=False)
sample_srs = population.loc[sample_srs_idx]
sample_prop = draw_stratified_sample(population, alloc_prop, rng)
sample_neyman = draw_stratified_sample(population, alloc_neyman, rng)

res_srs = srs_estimate(sample_srs, N)
res_prop = stratified_estimate(sample_prop, stratum_stats, N)
res_neyman = stratified_estimate(sample_neyman, stratum_stats, N)

one_run = pd.DataFrame([
    {"method": "SRS", **{k: res_srs[k] for k in ["estimate", "var_hat", "se", "df", "ci_low", "ci_high"]}},
    {"method": "Stratified-Proportional", **{k: res_prop[k] for k in ["estimate", "var_hat", "se", "df", "ci_low", "ci_high"]}},
    {"method": "Stratified-Neyman", **{k: res_neyman[k] for k in ["estimate", "var_hat", "se", "df", "ci_low", "ci_high"]}},
])

# Design effect: DEFF = V_complex(theta_hat) / V_SRS(theta_hat)  (same n)
srs_var = float(res_srs["var_hat"])
one_run["DEFF"] = one_run["var_hat"] / srs_var
print(one_run)


                    method    estimate     var_hat         se          df  \
0                      SRS  220.590000  222.823558  14.927276  299.000000   
1  Stratified-Proportional  238.505748  662.833302  25.745549   96.782493   
2        Stratified-Neyman  198.030773  107.350648  10.361016   28.172307   

       ci_low     ci_high      DEFF  
0  191.214171  249.965829  1.000000  
1  187.406509  289.604987  2.974700  
2  176.813040  219.248505  0.481774  



## 5) Monte Carlo simulation (B=1000)

For each design, track:
- estimate distribution
- empirical variance of estimator
- average theoretical variance estimate
- 95% CI coverage for true finite-population mean
- **Design effect (empirical):** \(\mathrm{DEFF} = V_{\text{complex}}(\hat{\bar{Y}}) / V_{\text{SRS}}(\hat{\bar{Y}})\) using simulation variances (same \(n\)); SRS row equals 1 by construction.
- **Per-replication DEFF (not one-shot):** each Monte Carlo draw yields \(\widehat{\mathrm{DEFF}}_b = \widehat{V}_{\text{complex},b}/\widehat{V}_{\text{SRS},b}\) from the usual variance estimators in that replication. Summarising \(b=1,\ldots,B\) gives a distribution (mean, SD, quantiles), not a single ratio from one sample.
- **Effective sample size:** \(n_{\mathrm{eff}} = n / \mathrm{DEFF}(\hat{\theta})\). Prefer \(\mathrm{DEFF}_{\mathrm{emp}}\) from simulation variances for \(n_{\mathrm{eff}}\). The **mean** of per-rep \(\widehat{\mathrm{DEFF}}_b\) can be distorted when those ratios have a heavy upper tail; the **median** of \(\widehat{\mathrm{DEFF}}_b\) is often closer to \(\mathrm{DEFF}_{\mathrm{emp}}\) in practice.

**Note on coverage:** Nominal 95% \(t\)-intervals target the *normal* sampling distribution of \(\bar{y}\). Heavy-tailed prices can produce **undercoverage** even when \(\widehat{\mathrm{Var}}\) is unbiased. Options: bootstrap / BCa intervals, transform (e.g. log), or report coverage as a **simulation diagnostic** rather than expecting exactly 0.95.


In [6]:

B = 1000
mc_result = run_monte_carlo(
    population,
    stratum_stats,
    alloc_prop,
    alloc_neyman,
    n_total=n_total,
    B=B,
    rng=rng,
    strata_col=STRATA_COL,
    y_col=Y_COL,
)
true_mean = mc_result.true_mean
mc = mc_result.mc
mc_summary = mc_result.mc_summary
deff_rep = mc_result.deff_rep
print_monte_carlo_summary(mc_result, n_total=n_total)


True population mean: 219.7561
                    method  empirical_var  avg_theoretical_var  coverage_95  \
0                      SRS    1341.083635          1301.281889        0.774   
1        Stratified-Neyman     627.773391           604.450117        0.845   
2  Stratified-Proportional    1194.475014          1189.852768        0.743   

   mean_estimate  DEFF_emp  n_eff_DEFF_emp  
0     219.650687  1.000000      300.000000  
1     218.832664  0.468109      640.876303  
2     217.006917  0.890679      336.821688  

Per-replication DEFF_hat = Vhat_strat / Vhat_SRS (B values per design; summary via describe):
       Stratified-Proportional  Stratified-Neyman
count              1000.000000        1000.000000
mean                  7.671835           4.342650
std                  36.665160           9.513245
min                   0.002333           0.003155
2.5%                  0.007255           0.008252
25%                   0.373734           0.517700
50%                   0.927

### Optional: SRS bootstrap CI coverage (diagnostic)

Percentile bootstrap on the SRS mean (resample **with replacement** within the SRS of size \(n\)) can track skewness better than a \(t\) interval. This is a **diagnostic** only; it does not replace the theory-based CIs above.

In [7]:
def bootstrap_srs_mean_ci(y: np.ndarray, n_boot: int, rng: np.random.Generator, alpha: float = 0.05):
    n = len(y)
    means = np.empty(n_boot)
    for i in range(n_boot):
        idx = rng.integers(0, n, size=n)
        means[i] = float(np.mean(y[idx]))
    lo = float(np.percentile(means, 100 * alpha / 2))
    hi = float(np.percentile(means, 100 * (1 - alpha / 2)))
    return lo, hi


BMC = 300
NBOOT = 400
true_mean = float(population[Y_COL].mean())
hits = 0
for _ in range(BMC):
    idx = rng.choice(population.index.to_numpy(), size=n_total, replace=False)
    y = population.loc[idx, Y_COL].to_numpy(dtype=float)
    lo, hi = bootstrap_srs_mean_ci(y, NBOOT, rng)
    if lo <= true_mean <= hi:
        hits += 1

print(f"SRS bootstrap (percentile) coverage of true mean: {hits / BMC:.3f} over {BMC} simulated samples (n_boot={NBOOT} each)")

SRS bootstrap (percentile) coverage of true mean: 0.793 over 300 simulated samples (n_boot=400 each)



## 6) Sensitivity analysis

Scenarios:
1. Main analysis (no cap)
2. Price capped at $5,000
3. Excluding IQR-flagged outliers (`price_outlier_iqr == True`)

For each scenario, run one n=300 comparison across SRS/proportional/Neyman.


In [8]:

sens = run_all_sensitivity_scenarios(
    population,
    rng,
    n_total=n_total,
    y_col=Y_COL,
    strata_col=STRATA_COL,
    outlier_flag_col=OUTLIER_FLAG_COL,
)
print(sens)


              scenario                   method    estimate         se  \
0                 main                      SRS  225.426667  18.889208   
1                 main  Stratified-Proportional  181.469218   6.894709   
2                 main        Stratified-Neyman  220.015827  14.379351   
3             cap_5000                      SRS  201.873333   8.247629   
4             cap_5000  Stratified-Proportional  220.051580  15.490997   
5             cap_5000        Stratified-Neyman  219.236229  11.654273   
6  exclude_iqr_flagged                      SRS  170.230000   5.033597   
7  exclude_iqr_flagged  Stratified-Proportional  166.163574   4.496934   
8  exclude_iqr_flagged        Stratified-Neyman  167.528896   4.523666   

       ci_low     ci_high  
0  188.254033  262.599300  
1  167.850542  195.087894  
2  191.115916  248.915738  
3  185.642580  218.104087  
4  189.348674  250.754486  
5  195.728341  242.744116  
6  160.324235  180.135765  
7  157.302031  175.025117  
8  158.


## 7) Math-to-code audit table


In [9]:

audit_table = pd.DataFrame([
    {
        "method": "SRS mean",
        "formula_used": "Ybar_hat = ybar",
        "code_function": "srs_estimate",
        "assumption": "SRS without replacement",
    },
    {
        "method": "SRS variance",
        "formula_used": "Var_hat(ybar) = (1-f)*s^2/n",
        "code_function": "srs_estimate",
        "assumption": "Finite population correction f=n/N",
    },
    {
        "method": "Stratified mean",
        "formula_used": "Ybar_st = sum_h W_h ybar_h",
        "code_function": "stratified_estimate",
        "assumption": "Independent SRS by stratum",
    },
    {
        "method": "Stratified variance",
        "formula_used": "sum_h W_h^2 (1-f_h) s_h^2 / n_h",
        "code_function": "stratified_estimate",
        "assumption": "Within-stratum sample variance used",
    },
    {
        "method": "Proportional allocation",
        "formula_used": "n_h ∝ N_h",
        "code_function": "proportional_allocation",
        "assumption": "Rounded with fixed minimum per stratum",
    },
    {
        "method": "Neyman allocation",
        "formula_used": "n_h ∝ N_h S_h",
        "code_function": "neyman_allocation",
        "assumption": "Uses stratum SD S_h from finite population",
    },
])

checks = {
    "N_matches_4702": int(N == 4702),
    "sum_Nh_equals_N": int(stratum_stats["N_h"].sum() == N),
    "sum_nh_prop_equals_n": int(alloc_prop.sum() == n_total),
    "sum_nh_neyman_equals_n": int(alloc_neyman.sum() == n_total),
    "f_in_0_1": int(0 <= res_srs["f"] <= 1),
    "has_scipy_for_t": int(HAVE_SCIPY),
}

print("Audit table")
print(audit_table)
print("\nSanity checks")
print(pd.Series(checks))


Audit table
                    method                     formula_used  \
0                 SRS mean                  Ybar_hat = ybar   
1             SRS variance      Var_hat(ybar) = (1-f)*s^2/n   
2          Stratified mean       Ybar_st = sum_h W_h ybar_h   
3      Stratified variance  sum_h W_h^2 (1-f_h) s_h^2 / n_h   
4  Proportional allocation                        n_h ∝ N_h   
5        Neyman allocation                    n_h ∝ N_h S_h   

             code_function                                  assumption  
0             srs_estimate                     SRS without replacement  
1             srs_estimate          Finite population correction f=n/N  
2      stratified_estimate                  Independent SRS by stratum  
3      stratified_estimate         Within-stratum sample variance used  
4  proportional_allocation      Rounded with fixed minimum per stratum  
5        neyman_allocation  Uses stratum SD S_h from finite population  

Sanity checks
N_matches_4702      